# Predict repeat orders with AdventureWorks on Snowflake

Which customers active in the past year will order again in the next 90 days? Train a Kumo Engine classifier in a Snowflake GPU notebook, publish its model, and score eligible customers.

Use `MODE="train"` for the first run or `MODE="reload"` with a recorded model version. Both routes rebuild the compatible graph and snapshot. Follow [Snowflake notebook setup](https://docs.nvidia.com/sdgm/kumo-engine/get-started-with-kumo-engine/installation/snowflake-notebook). Optional diagnostics follow the completed prediction workflow.

## Prepare your environment

Use a GPU Snowflake container notebook with supported local SSD scratch and PyPI access. Allow at least four times the materialized graph size in working storage, plus model/prediction artifacts; confirm actual disk capacity for your run.

Prepare `Customer`, `Product`, `SalesOrderHeader`, and `SalesOrderDetail` from the approved [Microsoft AdventureWorks sample](https://github.com/microsoft/sql-server-samples/tree/master/samples/databases/adventure-works). This notebook reads an existing extract; it does not load or alter source data. The role needs source-table read access, model-publication access, and permission to write the dedicated prediction destination. Optional diagnostic archival additionally needs an existing writable stage.


## Enter your configuration

Choose your warehouse, database, source schema, model name, and dedicated prediction output. Verify the example scoring date against your extract. Model reload additionally needs its recorded version.

Diagnostic inspection, plots, uploads, and downloads are opt-in. Set their switches only when needed; an archive stage is required only for upload. After a kernel restart, rerun configuration and continue at imports.

In [ ]:
from uuid import uuid4

from datetime import datetime

MODE = "train"  # "train" or "reload"
WAREHOUSE_NAME = "<warehouse>"
DATABASE_NAME = "<database>"
SCHEMA_NAME = "<schema>"
SCRATCH_DIR = "/tmp/kumoai-local"  # Confirm the backing disk.
MODEL_NAME = "ADVENTUREWORKS_CUSTOMER_ORDER_MODEL"
MODEL_VERSION = None  # Set the recorded version only for reload.
PREDICTION_TIME = datetime(2025, 6, 30)  # Check against your extract.
PREDICTION_OUTPUT_TABLE = ("ADVENTUREWORKS_ORDER_PREDICTIONS") + "_" + uuid4().hex[:8]
ARCHIVE_STAGE = "@<database>.<schema>.<archive-stage>/training-results"
SAVED_ARCHIVE = None  # None uses this session's uploaded archive.

# Enable optional diagnostics only when needed.
INSPECT_DIAGNOSTICS = False
PLOT_LEARNING_CURVES = False
WRITE_PREDICTIONS = True  # Replaces the dedicated output table's contents.
UPLOAD_DIAGNOSTICS = False
DOWNLOAD_DIAGNOSTICS = False
RUN_INDEX = None  # Automatically select this fit's unique new run, or set an index.
PLAN_INDEX = 0  # Inspect the first available trial plan; choose another if desired.

if MODE not in {"train", "reload"}:
    raise ValueError('Choose MODE = "train" or MODE = "reload".')

## Install

Install the Snowflake integration, restart the kernel, and rerun configuration before continuing.

In [ ]:
%pip install "kumo-engine[snowflake]"

<a id="runtime"></a>
## Import and configure scratch

After each kernel restart, run configuration and these cells. They do not depend on variables created during installation. Configure scratch before constructing any Kumo table, graph, or trainer.

In [ ]:
from pathlib import Path
import json
import shutil
import tempfile
from uuid import uuid4
import pandas as pd
from IPython.display import display
from snowflake.snowpark.context import get_active_session

session = get_active_session()
import kumo.engine as kumo
from kumoapi.typing import Dtype, Stype
from kumoapi.graph import Edge
from kumoapi.jobs import PredictionArtifactType, WriteMode

In [ ]:
scratch_root = Path(SCRATCH_DIR).expanduser().resolve()
scratch_root.mkdir(parents=True, exist_ok=True)
with tempfile.TemporaryFile(dir=scratch_root) as probe:
    probe.write(b"kumo scratch check")
    probe.flush()
kumo.configure(scratch_dir=str(scratch_root))
print("Kumo version:", kumo.__version__)
print("Scratch:", scratch_root)
print("Available GiB:", round(shutil.disk_usage(scratch_root).free / 2**30, 1))

import torch
if not torch.cuda.is_available():
    raise RuntimeError("Select the matching GPU notebook runtime and compute pool.")
for index in range(torch.cuda.device_count()):
    print("GPU:", torch.cuda.get_device_name(index))

The printed version identifies the installed build. The probe checks write access and free space at this path; it does not establish that the path is on supported local storage. Confirm the backing disk and capacity before materializing data.

## Connect and inspect the source

`from_spcs_env()` uses runtime-provided account and host settings and the mounted OAuth token. Never copy credentials into this notebook. The table list checks catalog visibility; column inspection checks source metadata access. Confirm all four tables appear. These checks do not prove model publication or output-write permissions.

In [ ]:
for name, value in {"warehouse": WAREHOUSE_NAME, "database": DATABASE_NAME,
                    "schema": SCHEMA_NAME}.items():
    if not value or "<" in value:
        raise ValueError(f"Set your {name} in configuration.")
connector = kumo.SnowflakeConnector.from_spcs_env(
    warehouse=WAREHOUSE_NAME, database=DATABASE_NAME, schema_name=SCHEMA_NAME,
)
print("Tables:", connector.table_names())
display(pd.DataFrame([
    {"column": column.name, "physical_type": column.dtype, "semantic_type": column.stype}
    for column in connector["SalesOrderHeader"].columns
]))

## Describe the four tables

A primary key identifies a row. A physical type describes its stored representation, such as a date; a semantic type describes its modeling role, such as a category or numerical value. `with_column()` returns an updated definition. Creating definitions can inspect or sample source data.

The customer and product definitions select the columns used in this example.

In [ ]:
customer = (
    kumo.Table.from_source_table(
        connector["Customer"],
        ["CustomerID", "StoreID", "TerritoryID"],
        primary_key="CustomerID",
    )
    .with_column("StoreID", stype=Stype.categorical)
    .with_column("TerritoryID", stype=Stype.categorical)
)

product = (
    kumo.Table.from_source_table(
        connector["Product"],
        ["ProductID", "Name", "Color", "ProductLine", "Class",
         "StandardCost", "ListPrice"],
        primary_key="ProductID",
    )
    .with_column("Name", stype=Stype.text)
    .with_column("Color", stype=Stype.categorical)
    .with_column("ProductLine", stype=Stype.categorical)
    .with_column("Class", stype=Stype.categorical)
    .with_column("StandardCost", stype=Stype.numerical)
    .with_column("ListPrice", stype=Stype.numerical)
)

Event timestamps let Kumo separate information available before a prediction boundary from future outcomes. Using future information as an input causes temporal leakage.

This sample uses `OrderDate` for orders and assumes `ModifiedDate` is an appropriate availability timestamp for line items, preserving the working example. A modification timestamp does not necessarily represent original availability in other datasets. Verify that assumption, including later edits to records, before adapting these definitions.

In [ ]:
sales_order_header = (
    kumo.Table.from_source_table(
        connector["SalesOrderHeader"],
        ["SalesOrderID", "CustomerID", "OrderDate", "OnlineOrderFlag",
         "TerritoryID", "SubTotal", "TotalDue"],
        primary_key="SalesOrderID",
        time_column="OrderDate",
    )
    .with_column("CustomerID", stype=Stype.ID)
    .with_column("OrderDate", dtype=Dtype.date)
    .with_column("OnlineOrderFlag", stype=Stype.categorical)
    .with_column("TerritoryID", stype=Stype.categorical)
    .with_column("SubTotal", stype=Stype.numerical)
    .with_column("TotalDue", stype=Stype.numerical)
)

In [ ]:
sales_order_detail = (
    kumo.Table.from_source_table(
        connector["SalesOrderDetail"],
        ["SalesOrderDetailID", "SalesOrderID", "ProductID", "ModifiedDate",
         "OrderQty", "UnitPrice", "LineTotal"],
        primary_key="SalesOrderDetailID",
        time_column="ModifiedDate",
    )
    .with_column("SalesOrderID", stype=Stype.ID)
    .with_column("ProductID", stype=Stype.ID)
    .with_column("ModifiedDate", dtype=Dtype.date)
    .with_column("OrderQty", stype=Stype.numerical)
    .with_column("UnitPrice", stype=Stype.numerical)
    .with_column("LineTotal", stype=Stype.numerical)
)

## Connect the tables and materialize a snapshot

A **graph** describes the tables and their relationships. Each arrow below goes from a foreign-key column to the destination table's primary key:

```text
SalesOrderDetail.ProductID    -> Product.ProductID
SalesOrderDetail.SalesOrderID -> SalesOrderHeader.SalesOrderID
SalesOrderHeader.CustomerID   -> Customer.CustomerID
```

Validation checks the graph definition; it is not a blanket guarantee of source-data quality. A **snapshot** materializes source data into the local graph index used for training and prediction. Snapshot creation reads the tables and consumes local disk. Reuse requires an unchanged, valid retained snapshot; loading a model does not restore a lost index.

In [ ]:
graph = kumo.Graph(
    tables=[customer, product, sales_order_header, sales_order_detail],
    edges=[
        Edge(src_table="SalesOrderHeader", fkey="CustomerID", dst_table="Customer"),
        Edge(src_table="SalesOrderDetail", fkey="SalesOrderID", dst_table="SalesOrderHeader"),
        Edge(src_table="SalesOrderDetail", fkey="ProductID", dst_table="Product"),
    ],
)

graph.validate()

## Inspect the graph

Inspect the declared tables and relationships before training.
Graph visualization is a commented preview because this public release does not expose it.

In [ ]:
# graph.visualize(show_cols=True)

In [ ]:
print("graph id:", graph.id)

In [ ]:
snapshot = graph.snapshot()
print("Snapshot:", snapshot.id)
print("Local index:", snapshot.index_path)

## Define the prediction question

A **predictive query** states the entity, future outcome, and population. This query scores customers with at least one order in the previous 365 days and predicts whether they will order in the next 90 days. It does not necessarily include every customer. `> 0` makes the target binary.

In [ ]:
pquery = kumo.PredictiveQuery(
    graph=graph,
    query="""
        PREDICT COUNT(SalesOrderHeader.*, 0, 90, days) > 0
        FOR EACH Customer.CustomerID
        WHERE COUNT(SalesOrderHeader.*, -365, 0, days) > 0
    """,
)

pquery.validate()
print("Task type:", pquery.get_task_type())

## Train a new model or reload a recorded version

For `MODE = "train"`, a **model plan** controls architecture, sampling, and optimization. Inspect the suggested plan and configured validation selection metric. The suggested FAST plan supplies the training defaults. Do not assume the metric's value or direction. Compare like-for-like validation metrics and reserve test metrics for evaluating the selected model.

This walkthrough does not configure a text encoder for `Product.Name`. If Kumo reports that the column will not be encoded, the run proceeds without that feature. GloVe is not supported.

For `MODE = "reload"`, these training cells skip automatically. Use the recorded model name/version from a trusted registry publication, with compatible graph and query definitions. Reload restores the trained model and generation settings, not source data or snapshot files.

In [ ]:
from kumoapi.model_plan import RunMode

if MODE == "train":
    model_plan = pquery.suggest_model_plan(snapshot, run_mode=RunMode.FAST)
    print(model_plan)
    print("Validation selection metric:", getattr(
        model_plan.training_job, "tune_metric", "not exposed by this build"))
else:
    if not MODEL_VERSION or "<" in MODEL_VERSION:
        raise ValueError("Set the exact recorded model version before reloading.")
    trainer = kumo.Trainer.load_from_connector(
        connector,
        location=kumo.ModelRegistryLocation(MODEL_NAME, version=MODEL_VERSION),
    )
    print("Loaded:", MODEL_NAME, MODEL_VERSION)

### Generate historical examples

Generation computes historical populations, labels, and splits before fitting. At each historical boundary, past information supplies inputs and the subsequent 90 days supply the target. The later scoring date is a separate boundary; historical outcome evaluation requires the relevant follow-up data.

The next cell creates a training table only on the first-run route.

In [ ]:
if MODE == "train":
    train_table = pquery.generate_training_table(snapshot)
    print("Training table:", train_table.train_table_path)
else:
    print("Reload route: no training table is generated.")

### Fit and publish once

On the first-run route, this cell trains and publishes the selected model to your configured Snowflake database/schema in the same call. It requires model-publication permissions. Record the returned name and version for future reloads. Running the cell again starts another fit.

For a deliberately local-only experiment, set `model_output = None` in this cell before running it. Retain the complete resulting bundle, including manifests and metadata. You can use `kumo.Trainer.load(bundle_path)` only while all required files remain available.

During fit, the cell hides the two known Optuna notices and the pandas incompatible-dtype FutureWarning. This reduces output noise; it does not repair the underlying pandas assignment. Other warnings remain visible, and the filters are restored after fit.

In [ ]:
if MODE == "train":
    runs_before_fit = {p.parent for p in scratch_root.rglob("auto_train_progress.json")}
    trainer = kumo.Trainer(model_plan)
    model_output = kumo.ModelOutputConfig(
        connector, location=kumo.ModelRegistryLocation(MODEL_NAME),
    )
    import warnings
    from optuna.exceptions import ExperimentalWarning

    # Hide only these known notices during fit; preserve other warnings.
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message=r"`warn_independent_sampling` has been deprecated.*",
            category=FutureWarning,
        )
        warnings.filterwarnings(
            "ignore",
            message=r"Argument ``multivariate`` is an experimental feature.*",
            category=ExperimentalWarning,
        )
        warnings.filterwarnings(
            "ignore",
            message=r"Setting an item of incompatible dtype is deprecated.*",
            category=FutureWarning,
        )
        training_result = trainer.fit(
            train_table, snapshot=snapshot, export_config=model_output,
        )
    published = training_result.published_model
    if published is not None:
        print("Record registry:", DATABASE_NAME, SCHEMA_NAME)
        print("Record model name/version:", published.name, published.version)
    else:
        print("Local-only model. Retain complete bundle:", training_result.bundle_dir)
else:
    print("Reload route: fit and publication skipped.")

### Evaluate the selected model

Review validation and test metrics separately. AUROC measures repeat-order ranking; precision and recall describe the decision tradeoff. The published model retains model state; diagnostic artifacts are optional and appear after scoring.

In [ ]:
if MODE == "train":
    print("Available metrics:", training_result.metrics)
    print("Reusable bundle:", training_result.bundle_dir)
    for name, path in training_result.artifact_paths.items():
        print(name, path)
else:
    print("Model loaded. Earlier experiment diagnostics require retained scratch or an archive.")

## Check the scoring date

Both routes now have a fitted trainer and a compatible graph snapshot. Compare the extract’s order-date range with the configured anchor before preparing the customer population. A future prediction date does not provide observed outcomes for evaluation.

In [ ]:
from snowflake.snowpark.functions import col, min as sf_min, max as sf_max

def quoted_identifier(name):
    return '"' + name.replace('"', '""') + '"'

order_dates = session.table([
    quoted_identifier(name)
    for name in [DATABASE_NAME, SCHEMA_NAME, "SalesOrderHeader"]
])
display(order_dates.select(
    sf_min(col('"OrderDate"')).alias("FIRST_ORDER"),
    sf_max(col('"OrderDate"')).alias("LAST_ORDER"),
).collect())
print("Configured example boundary:", PREDICTION_TIME)

## Batch predictions

For this temporal query, use a timezone-naive timestamp interpreted as UTC. Reuse the fitted trainer's saved training-table generation plan. The generated population contains customers meeting the past-year restriction at that date. This timestamp example does not apply to static queries.

In [ ]:
prediction_table = pquery.generate_prediction_table(
    snapshot,
    prediction_time=PREDICTION_TIME,
    plan=trainer.training_table_generation_plan,
)

### Score locally, with an optional Snowflake output

**`WRITE_PREDICTIONS = True` authorizes `OVERWRITE`, which replaces the configured destination table's contents.** Choose a dedicated table you own, never a source table, and review the configuration before running this cell. Writing is enabled by default; set `WRITE_PREDICTIONS = False` to keep results only in scratch. Select a different output name to retain an earlier batch.

The binary threshold is 0.5, matching this example. Changing the threshold changes labels, not probabilities.

In [ ]:
prediction_output = None
if WRITE_PREDICTIONS:
    import re
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", PREDICTION_OUTPUT_TABLE):
        raise ValueError("Choose a simple dedicated output table name in configuration.")
    source_names = {"CUSTOMER", "PRODUCT", "SALESORDERHEADER", "SALESORDERDETAIL"}
    if PREDICTION_OUTPUT_TABLE.upper() in source_names:
        raise ValueError("Prediction output must not replace a source table.")
    prediction_output = kumo.PredictionOutputConfig(
        connector=connector, artifact_type=PredictionArtifactType.PREDICTIONS,
        table_name=PREDICTION_OUTPUT_TABLE, write_mode=WriteMode.OVERWRITE,
    )


### Run batch prediction

Score the prepared customer population with the retained model.


In [ ]:
prediction_result = trainer.predict(
    prediction_table, snapshot=snapshot,
    binary_classification_threshold=0.5, output_config=prediction_output,
)


### Inspect and save results

Expect one row for each eligible customer. `True_PROB` is the 90-day repeat-order probability; `TARGET_PRED` is true above the 0.5 threshold. Verify IDs and row counts, preview ten rows, and save a local Parquet copy. This small example collects the full result; retain the file outside temporary runtime storage.

In [ ]:
import pandas as pd
from pathlib import Path
from uuid import uuid4

population = pd.read_parquet(prediction_table.table_path)
assert len(population) >= 2, "Choose at least two scoring rows."
scoring_rows = population[[name for name in ("ENTITY1", "TIMESTAMP") if name in population]]
assert not any(str(column).startswith("TARGET") for column in scoring_rows)
batch_scores = prediction_result.frame
assert set(batch_scores["ENTITY"]) == set(population["ENTITY1"])
assert len(batch_scores) == len(population)
assert batch_scores.ENTITY.is_unique
print(batch_scores.head(10).to_string(index=False))
saved_predictions = Path.cwd() / ("batch-predictions-" + uuid4().hex + ".parquet")
batch_scores.to_parquet(saved_predictions, index=False)
print("Saved:", saved_predictions)

## Reuse the model

Record the published model name/version and database/schema. In a later session use `MODE="reload"`, configure scratch, reconnect, and rebuild a compatible graph and snapshot before scoring. The model does not contain the source data or snapshot. See [model reuse](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/train-a-model/publish-and-retrieve-models).

<a id="diagnostics"></a>
## Optional: inspect experiments and explanations

Set `INSPECT_DIAGNOSTICS=True` to enable inspection. Nothing here is required for prediction. Discovery lists retained AutoML directories. The notebook automatically selects a run only when exactly one new directory appeared during this fit. It never assumes the latest directory is this model's run.

If selection is ambiguous, set `RUN_INDEX` in configuration to an index from the list and rerun selection. On reload, select a retained run explicitly; loading a registry model does not restore its diagnostics. Set `INSPECT_DIAGNOSTICS = False` to skip inspection. Plotting has its own switch.

In [ ]:
automl_dirs = sorted({
    path.parent for path in scratch_root.rglob("auto_train_progress.json")
    if path.is_file()
})
for index, path in enumerate(automl_dirs):
    print(index, path)
if not automl_dirs:
    print("No retained AutoML runs found under configured scratch.")

In [ ]:
automl_dir = None
if INSPECT_DIAGNOSTICS:
    if RUN_INDEX is not None:
        if not isinstance(RUN_INDEX, int) or not 0 <= RUN_INDEX < len(automl_dirs):
            raise ValueError("Choose one of the displayed run indices.")
        automl_dir = automl_dirs[RUN_INDEX]
    elif MODE == "train" and "runs_before_fit" in globals():
        new_runs = [path for path in automl_dirs if path not in runs_before_fit]
        if len(new_runs) == 1:
            automl_dir = new_runs[0]
    if automl_dir is None:
        print("No unambiguous run selected. Set RUN_INDEX from the displayed list.")
    else:
        print("Selected:", automl_dir)
else:
    print("Diagnostic inspection disabled.")

### Read progress and compare trials

These small readers report missing or unreadable files without blocking prediction. The progress summary shows completed/total trials and each trial's recorded epochs, elapsed seconds, and accelerator. Missing optional values mean they were not recorded.

In [ ]:
def read_json(path):
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (OSError, ValueError) as error:
        print(f"Cannot read {path.name}: {error}")
        return {}

def read_csv(path):
    try:
        return pd.read_csv(path)
    except (OSError, pd.errors.EmptyDataError, pd.errors.ParserError) as error:
        print(f"Cannot read {path.name}: {error}")
        return pd.DataFrame()

def history_frames(progress):
    summaries, curves = [], []
    for trial, details in (progress.get("trial_progress") or {}).items():
        details = details or {}
        summaries.append({"trial": str(trial),
                          "epochs_completed": details.get("num_epochs_completed"),
                          "elapsed_seconds": details.get("elapsed_time"),
                          "accelerator": details.get("accelerator")})
        for epoch, metrics in (details.get("metrics") or {}).items():
            try:
                epoch_number = int(epoch)
            except (TypeError, ValueError):
                print("Skipping unrecognized epoch:", epoch)
                continue
            metrics = metrics or {}
            curves.append({"trial": str(trial), "epoch": epoch_number,
                           **(metrics.get("train_metrics") or {}),
                           **(metrics.get("validation_metrics") or {})})
    summary = pd.DataFrame(summaries)
    history = pd.DataFrame(curves)
    if not history.empty:
        history = history.sort_values(["trial", "epoch"])
    return summary, history

In [ ]:
progress = read_json(automl_dir / "auto_train_progress.json") if automl_dir else {}
trial_summary, learning_curves = history_frames(progress)
print("Completed / total trials:", progress.get("completed_trials", "not recorded"),
      "/", progress.get("total_trials", "not recorded"))
if not trial_summary.empty:
    display(trial_summary)
else:
    print("No trial progress recorded or no run selected.")
if not learning_curves.empty:
    display(learning_curves)
print("Available epoch metrics:", [
    c for c in learning_curves.columns if c not in {"trial", "epoch"}
])

In [ ]:
trial_frames = []
for path in sorted(automl_dir.glob("version_*/val_result.csv")) if automl_dir else []:
    frame = read_csv(path)
    if not frame.empty:
        frame["trial_directory"] = path.parent.name
        trial_frames.append(frame)
if trial_frames:
    display(pd.concat(trial_frames, ignore_index=True))
else:
    print("No completed trial validation results found.")

plan_paths = sorted(automl_dir.glob("version_*/model_plan.json")) if automl_dir else []
for index, path in enumerate(plan_paths):
    print(index, path.relative_to(automl_dir))

The first available trial plan is displayed by default. Change `PLAN_INDEX` in configuration to inspect another. Compare trial validation results using the configured selection metric, not an assumed metric or direction.

In [ ]:
if plan_paths and PLAN_INDEX is not None:
    if not isinstance(PLAN_INDEX, int) or not 0 <= PLAN_INDEX < len(plan_paths):
        raise ValueError("Choose a displayed plan index.")
    display(read_json(plan_paths[PLAN_INDEX]))
else:
    print("No trial plan selected.")

### Plot available learning curves

Select exact metric names printed above. The default tries training and validation loss. Compare how validation changes while training improves. Divergence can suggest overfitting; nearly perfect results may warrant a leakage check. These are diagnostic signals, not proof of a cause. Plotting is optional and does not affect other inspection cells.

In [ ]:
METRIC_COLUMNS = ["train_loss", "val_loss"]
columns = [name for name in METRIC_COLUMNS if name in learning_curves.columns]
if not PLOT_LEARNING_CURVES:
    print("Learning-curve plotting disabled.")
elif learning_curves.empty or not columns:
    print("No matching completed epoch metrics. Choose from the available names.")
else:
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        print("Matplotlib is unavailable. Use the displayed epoch table instead.")
    else:
        figure, axis = plt.subplots(figsize=(9, 5))
        for trial, rows in learning_curves.groupby("trial", sort=True):
            for column in columns:
                values = pd.to_numeric(rows[column], errors="coerce")
                if values.notna().any():
                    axis.plot(rows["epoch"], values, marker="o", label=f"{trial}: {column}")
        axis.set(xlabel="Epoch", ylabel="Metric", title="Learning curves")
        axis.grid(alpha=0.25)
        if axis.lines:
            axis.legend()
        plt.show()

### Inspect gradient scores and column analysis

Supported training tasks produce post-fit gradient scores and column analysis when explanation is enabled. Availability depends on task, model, and build, and explanations can be disabled through the model plan's `training_job.disable_explain` setting.

Gradient scores describe model sensitivity to input columns, not causal effects. Column analysis groups feature values into cohorts and summarizes predictions and outcomes. Explanation errors may be logged while the model remains usable. Missing scores do not prove a column was excluded from training.

The following cells search beneath the selected run, display the actual artifact paths, and read what exists. They do not use a separate explanation API or require the plotting cell.

In [ ]:
score_paths = sorted(automl_dir.rglob("explain_test/scores.csv")) if automl_dir else []
if not score_paths:
    print("No gradient scores. Check selected run, build, explanation setting, and logs.")
for path in score_paths:
    print("Scores:", path.relative_to(automl_dir))
    scores = read_csv(path)
    if "score" in scores.columns:
        scores = scores.sort_values("score", ascending=False)
    display(scores)

In [ ]:
analysis_paths = sorted(automl_dir.rglob(
    "column_analysis/test/column_analysis.json"
)) if automl_dir else []
if not analysis_paths:
    print("No column analysis found for the selected run.")
for path in analysis_paths:
    print("Column analysis:", path.relative_to(automl_dir))
    analysis = read_json(path)
    display(pd.json_normalize(analysis.get("cohorts") or []))

### Locate other diagnostics

When present, `best_trainer.json` identifies the selected trial; `train_progress.json` records individual trial progress; TensorBoard event files support deeper curve inspection; `failed_trials` contains failure diagnostics; and `analyze` contains optimization analysis. Parameter importance in optimization analysis concerns training settings, not input-column explanations. Not every run emits every artifact type.

In [ ]:
if automl_dir:
    for pattern in ["best_trainer.json", "version_*/train_progress.json",
                    "**/events.out.tfevents.*", "failed_trials/*", "analyze/*"]:
        paths = sorted(automl_dir.glob(pattern))
        print(pattern, ":", len(paths), "matches")
        for path in paths[:20]:
            print(" ", path.relative_to(automl_dir))

## Optional: preserve diagnostics

Archive only the selected AutoML directory to an existing restricted internal stage. Archives can contain large checkpoints and sensitive diagnostic information. Allow additional local space for the ZIP while the original files remain. This is separate from model publication: an archive is not a registry entry, source-data backup, or snapshot restore mechanism.

Set `UPLOAD_DIAGNOSTICS=True` and configure the existing archive stage to upload diagnostics. A selected run is required. The unique filename avoids replacing earlier archives. This cell checks upload results and prints the exact path to record. It does not delete scratch.

In [ ]:
uploaded_archive_stage_path = None
if UPLOAD_DIAGNOSTICS and INSPECT_DIAGNOSTICS:
    if automl_dir is None:
        raise ValueError("Select a run in optional diagnostics first.")
    if not ARCHIVE_STAGE or "<" in ARCHIVE_STAGE or not ARCHIVE_STAGE.startswith("@"):
        raise ValueError("Configure an existing writable internal archive stage.")
    archive_dir = Path(tempfile.mkdtemp(prefix="diagnostic-export-", dir=scratch_root))
    archive_name = f"adventureworks-{automl_dir.name}-{uuid4().hex}"
    archive_path = shutil.make_archive(str(archive_dir / archive_name), "zip", root_dir=automl_dir)
    results = session.file.put(
        f"file://{archive_path}", ARCHIVE_STAGE, overwrite=False, auto_compress=False,
    )
    for result in results:
        print(result.target, result.status, result.message)
    if not results or any(result.status != "UPLOADED" for result in results):
        raise RuntimeError("Archive upload incomplete. Review results; local files remain.")
    uploaded_archive_stage_path = f"{ARCHIVE_STAGE.rstrip('/')}/{Path(archive_path).name}"
    print("Record archive path:", uploaded_archive_stage_path)
else:
    print("Archive upload skipped.")

### Retrieve and inspect an archive

Set `DOWNLOAD_DIAGNOSTICS=True` to retrieve the archive at the exact path returned by this session’s upload. To inspect a previous archive, set `SAVED_ARCHIVE` in configuration to its recorded full stage path. If upload is disabled, a saved path is required. Set `DOWNLOAD_DIAGNOSTICS = False` to skip retrieval.

This cell uses the active Snowpark session and a fresh directory. It lists ZIP contents without extracting or executing them. Locate validation CSVs and plan JSON by their listed paths. Local files last only as long as the environment retains its filesystem.

In [ ]:
if DOWNLOAD_DIAGNOSTICS:
    from zipfile import ZipFile
    archive_to_download = SAVED_ARCHIVE or globals().get("uploaded_archive_stage_path")
    if not archive_to_download:
        raise ValueError("Set SAVED_ARCHIVE or enable upload and select a run first.")
    if "<" in archive_to_download or not archive_to_download.endswith(".zip"):
        raise ValueError("Set the exact saved archive stage path in configuration.")
    download_dir = Path(tempfile.mkdtemp(prefix="diagnostic-download-", dir=scratch_root))
    downloads = session.file.get(archive_to_download, str(download_dir))
    downloaded_zip = download_dir / archive_to_download.rsplit("/", 1)[-1]
    if not downloads or not downloaded_zip.is_file():
        raise FileNotFoundError("Archive download failed. Check stage access and path.")
    with ZipFile(downloaded_zip) as archive:
        members = archive.infolist()
        print("Members:", len(members))
        for member in members[:50]:
            print(member.filename, member.file_size, "bytes")
    print("Downloaded:", downloaded_zip)
else:
    print("Archive retrieval skipped.")

## Finish

Retain the published model and any prediction exports you need. Optional archives have a separate lifecycle; this notebook leaves source tables and retained artifacts intact.
